In [1]:
import os
print(os.listdir('checkpoints'))

['loss_history.csv', 'loss_history.json', 'loss_plot.png', 'restormer_best_model.pth', 'restormer_last_checkpoint.pth', 'scunet_last_checkpoint.pth', 'unet_best_model.pth', 'unet_last_checkpoint.pth']


In [2]:
import os
import cv2
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Устройство инференса: {device}")

# ⚠️ УКАЖИТЕ ПУТИ:
NOISY_DIR = "rec_00301.tif"          # Папка с грязными снимками Kaggle
CHECKPOINT_PATH = "restormer_best_model.pth"        # Путь к весам (SCUNet или Restormer)

# 1. Загрузка модели (пример для SCUNet)
from models.network_scunet import SCUNet
model = SCUNet(in_nc=1, config=[4, 4, rec_00301.tif4, 4, 4, 4, 4], dim=64).to(device)

if os.path.exists(CHECKPOINT_PATH):
    state_dict = torch.load(CHECKPOINT_PATH, map_location=device)
    if 'model_state_dict' in state_dict:
        model.load_state_dict(state_dict['model_state_dict'])
    else:
        model.load_state_dict(state_dict)
    print(f"✅ Успешно загружены веса из: '{CHECKPOINT_PATH}'")
else:
    print(f"⚠️ Чекпоинт {CHECKPOINT_PATH} не найден!")

model.eval()

# Список файлов
noisy_files = sorted([f for f in os.listdir(NOISY_DIR) if f.lower().endswith(('.tif', '.tiff'))])
print(f"📁 Найдено снимков: {len(noisy_files)} шт.")

✅ Устройство инференса: cpu


ModuleNotFoundError: No module named 'models'

In [ ]:
def denoise_patches_only(
    model, 
    image_path, 
    patch_size=512, 
    overlap_ratio=0.2, 
    batch_size=8, 
    device=device
):
    """
    1. Читает снимок
    2. Нарезает на патчи во временную память
    3. Прогоняет патчи через модель
    4. Возвращает списки: (грязные патчи, очищенные патчи, координаты)
    """
    raw_img = tiff.imread(image_path).astype(np.float32)
    h, w = raw_img.shape
    
    # Нормализация
    img_min, img_max = np.min(raw_img), np.max(raw_img)
    norm_img = (raw_img - img_min) / (img_max - img_min + 1e-8)

    # Расчет шага
    stride = int(patch_size * (1.0 - overlap_ratio))
    y_steps = list(range(0, h - patch_size + 1, stride))
    if y_steps[-1] != h - patch_size: y_steps.append(h - patch_size)

    x_steps = list(range(0, w - patch_size + 1, stride))
    if x_steps[-1] != w - patch_size: x_steps.append(w - patch_size)

    # 1. Извлекаем сырые патчи
    noisy_patches_list = []
    coords_list = []
    for y in y_steps:
        for x in x_steps:
            noisy_patches_list.append(norm_img[y:y+patch_size, x:x+patch_size])
            coords_list.append((y, x))

    total_p = len(noisy_patches_list)
    cleaned_patches_list = []

    # 2. Прогон через модель батчами
    with torch.no_grad():
        for i in range(0, total_p, batch_size):
            batch_np = np.array(noisy_patches_list[i:i+batch_size])
            batch_tensor = torch.from_numpy(batch_np).unsqueeze(1).float().to(device)

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                preds = model(batch_tensor)
            
            preds_np = np.clip(preds.squeeze(1).cpu().numpy(), 0.0, 1.0)
            for p in preds_np:
                cleaned_patches_list.append(p)

    return noisy_patches_list, cleaned_patches_list, coords_list

print("✅ Функция denoise_patches_only готова к работе!")

In [ ]:
# Выбираем любой один снимок для демонстрации
sample_file = noisy_files[0]
sample_path = os.path.join(NOISY_DIR, sample_file)

print(f"🎯 Обработка патчей для снимка: '{sample_file}'...")

# 1. Получаем списки патчей
noisy_patches, cleaned_patches, coords = denoise_patches_only(
    model=model,
    image_path=sample_path,
    patch_size=512,
    overlap_ratio=0.2,
    batch_size=8,
    device=device
)

# 2. Сколько патчей вывести на экран (например, 6 штук или len(noisy_patches) для всех 64)
NUM_PATCHES_TO_SHOW = 6  

fig, axes = plt.subplots(NUM_PATCHES_TO_SHOW, 3, figsize=(16, NUM_PATCHES_TO_SHOW * 5))

for i in range(NUM_PATCHES_TO_SHOW):
    n_p = noisy_patches[i]
    c_p = cleaned_patches[i]
    y, x = coords[i]
    
    # Карта шума = Входной патч МИНУС Очищенный патч
    noise_map = n_p - c_p

    # Колонка 1: Грязный патч
    axes[i, 0].imshow(n_p, cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_title(f"Патч #{i} (y={y}, x={x})\n1. Грязный вход (Kaggle)", fontsize=11)
    axes[i, 0].axis('off')

    # Колонка 2: Очищенный патч нейросетью
    axes[i, 1].imshow(c_p, cmap='gray', vmin=0, vmax=1)
    axes[i, 1].set_title(f"Патч #{i} (y={y}, x={x})\n2. Очищенный патч (Выход модели)", fontsize=11, color='darkgreen')
    axes[i, 1].axis('off')

    # Колонка 3: Извлеченный шум
    axes[i, 2].imshow(noise_map, cmap='coolwarm')
    axes[i, 2].set_title(f"Патч #{i} (y={y}, x={x})\n3. Удаленный шум КТ", fontsize=11, color='darkblue')
    axes[i, 2].axis('off')

plt.suptitle(f"📊 ПАТЧИ ПОСЛЕ ОЧИСТКИ НЕЙРОСЕТЬЮ (Снимок: '{sample_file}')", fontsize=15, y=1.001)
plt.tight_layout()
plt.show()